# 选修E1 · Day 3：多Agent系统设计 · 上机练习（v5.0）

> **真实库**：LangGraph（多Agent协作图）+ networkx（拓扑分析）
> **核心任务**：4个营销Agent（researcher/strategist/writer/reviewer）协作完成营销策略产出
> **拓扑对比**：supervisor中心化 vs team去中心化
> **营销映射**：透肌精华竞品分析（雅诗兰黛）+ 策略 + 文案 + 审核，多Agent涌现团队决策

本笔记本构建一个完整的多Agent营销协作系统。你将：
1. 定义Agent间通信协议（AgentMessage）和共享状态（MultiAgentState）
2. 实现4个营销Agent节点（researcher/strategist/writer/reviewer）
3. 用LangGraph构建supervisor中心化拓扑和team去中心化拓扑
4. 用networkx分析两种拓扑的中心性/连通性/瓶颈
5. 运行双拓扑系统，分析涌现行为，映射天道推演

> 天道推演视角：多Agent系统是"可计算沙盘"--supervisor模拟决策者，4个Agent模拟利益相关方，networkx量化推演拓扑质量


In [ ]:
# === 导入真实库 ===
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from enum import Enum
import operator
import networkx as nx
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# === 真实营销数据（基于护肤品电商场景，复用Day 1/2）===
PRODUCT_DB = {
    "透肌精华": "透肌焕亮精华液，299元，主打美白焕亮，含烟酰胺3%+维C衍生物，目标用户25-35岁都市白领。",
    "玻尿酸面霜": "玻尿酸保湿面霜，159元，主打深层补水，含双重玻尿酸，目标用户18-30岁女性。",
}
COMPETITOR_DB = {
    "雅诗兰黛": "雅诗兰黛小棕瓶精华，760元/30ml，市场占有率18%，优势：品牌力强、渠道完善；劣势：价格高、年轻化不足。",
    "兰蔻": "兰蔻小黑瓶精华，780元/30ml，市场占有率15%，优势：科技感强、专柜体验；劣势：下沉市场覆盖弱。",
}

# === 统一营销任务（所有Agent协作完成同一个任务）===
MARKETING_TASK = "为透肌精华制定营销策略，竞品分析雅诗兰黛，产出合规文案"

# === 离线模拟LLM（无需API Key，预编排Agent响应）===
class StubChatModel(BaseChatModel):
    """离线模拟LLM，预编排Agent响应序列，保证无API Key可运行。
    替换为ChatOpenAI/ChatAnthropic即可使用真实LLM驱动多Agent涌现。"""
    responses: list = []
    call_index: int = 0

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        idx = self.call_index
        self.call_index += 1
        if idx < len(self.responses):
            resp = self.responses[idx]
        else:
            resp = AIMessage(content="Agent任务完成。")
        return ChatResult(generations=[ChatGeneration(message=resp)])

    @property
    def _llm_type(self):
        return "stub"

print("真实库导入成功")
print(f"  LangGraph: StateGraph + add_conditional_edges (多Agent协作图)")
print(f"  networkx: {nx.__version__} (Agent通信拓扑分析)")
print(f"  产品库: {list(PRODUCT_DB.keys())}")
print(f"  竞品库: {list(COMPETITOR_DB.keys())}")
print(f"  统一营销任务: {MARKETING_TASK}")
print(f"  StubChatModel: 离线模式（无API Key可运行）")

---
## TODO1：定义Agent间通信协议和共享状态

多Agent系统的核心是**通信**和**状态共享**。需要定义：

1. **MessageType枚举**：任务分配/结果汇报/信息共享/请求帮助/反馈/投票
2. **AgentMessage（pydantic BaseModel）**：sender/receiver/message_type/content/metadata/reply_to
3. **MultiAgentState（TypedDict）**：所有Agent共享的全局状态，包含task/messages/research_data/strategy/content/review_result/current_agent/revision_count/approved

这是A2A协议（Agent间互操作）的简化实现。真实A2A协议还包含Agent Card发现和任务状态查询，本Day聚焦消息格式层。

> 天道推演对应：AgentMessage是"因果链追踪"的载体，MultiAgentState是"沙盘"的当前状态


In [ ]:
# TODO1: 定义Agent间通信协议和共享状态

class MessageType(Enum):
    TASK_ASSIGNMENT = "task_assignment"
    RESULT_REPORT = "result_report"
    INFO_SHARING = "info_sharing"
    HELP_REQUEST = "help_request"
    FEEDBACK = "feedback"
    VOTE = "vote"

class AgentMessage(BaseModel):
    sender: str
    receiver: str  # "broadcast"表示广播
    message_type: MessageType
    content: str
    metadata: dict = Field(default_factory=dict)
    reply_to: str = None

class MultiAgentState(TypedDict):
    task: str
    messages: Annotated[list, operator.add]  # AgentMessage累积传递
    research_data: str
    strategy: str
    content: str
    review_result: str
    current_agent: str
    revision_count: int
    approved: bool

# 测试通信协议
test_msg = AgentMessage(
    sender="supervisor",
    receiver="researcher",
    message_type=MessageType.TASK_ASSIGNMENT,
    content="调研透肌精华竞品雅诗兰黛的市场份额和定价策略",
    metadata={"task_id": "mkt_001", "priority": "high"}
)
print("AgentMessage协议定义成功")
print(f"  消息类型: {[mt.name for mt in MessageType]}")
print(f"  测试消息: {test_msg.sender} -> {test_msg.receiver} [{test_msg.message_type.value}]")
print(f"  消息内容: {test_msg.content}")
print(f"  MultiAgentState: 9个共享字段（task/messages/research_data/strategy/content/...）")
print(f"  messages字段: Annotated[list, operator.add] 实现Agent间消息累积")

---
## TODO2：实现4个营销Agent节点函数

每个Agent是一个节点函数：接收State，返回State更新。4个Agent各有职责：

| Agent | 职责 | 读State | 写State |
|-------|------|---------|---------|
| researcher | 市场调研 | task | research_data + messages |
| strategist | 策略制定 | research_data | strategy + messages |
| writer | 文案生成 | strategy | content + messages |
| reviewer | 合规审核 | content | review_result + approved + messages |

每个Agent执行后向State追加AgentMessage（message_type=RESULT_REPORT），实现通信可追溯。

> 天道推演对应：researcher=局势感知，strategist=沙盘模拟，writer=最优路径推荐，reviewer=反馈学习


In [ ]:
# TODO2: 实现4个营销Agent节点函数

def researcher_agent(state: MultiAgentState) -> dict:
    """市场调研Agent：查询产品库和竞品库，产出市场分析"""
    product_info = PRODUCT_DB.get("透肌精华", "未找到产品信息")
    competitor_info = COMPETITOR_DB.get("雅诗兰黛", "未找到竞品信息")
    research_data = f"【市场调研报告】\n产品: {product_info}\n竞品: {competitor_info}"
    msg = AgentMessage(
        sender="researcher", receiver="supervisor",
        message_type=MessageType.RESULT_REPORT,
        content=f"完成市场调研：透肌精华299元 vs 雅诗兰黛760元，价格优势明显",
        metadata={"confidence": 0.9}
    )
    return {"research_data": research_data, "messages": [msg], "current_agent": "researcher"}

def strategist_agent(state: MultiAgentState) -> dict:
    """策略制定Agent：基于调研产出差异化策略"""
    research = state.get("research_data", "")
    strategy = "【内容策略】差异化定位：主打性价比+年轻化。核心主张'同等功效，一半价格'。目标用户25-35岁都市白领，渠道聚焦小红书+抖音。"
    msg = AgentMessage(
        sender="strategist", receiver="supervisor",
        message_type=MessageType.RESULT_REPORT,
        content="策略已制定：差异化定位，主打性价比+年轻化",
        metadata={"strategy_type": "differentiation"}
    )
    return {"strategy": strategy, "messages": [msg], "current_agent": "strategist"}

def writer_agent(state: MultiAgentState) -> dict:
    """文案生成Agent：基于策略写营销文案"""
    strategy = state.get("strategy", "")
    content = "【营销文案】透肌焕亮精华液，299元享同等美白功效。含烟酰胺3%+维C衍生物，媲美760元小棕瓶。年轻肌的聪明选择，美白不交智商税。"
    msg = AgentMessage(
        sender="writer", receiver="supervisor",
        message_type=MessageType.RESULT_REPORT,
        content="文案已生成：主打性价比，'美白不交智商税'",
        metadata={"word_count": 42}
    )
    return {"content": content, "messages": [msg], "current_agent": "writer"}

def reviewer_agent(state: MultiAgentState) -> dict:
    """合规审核Agent：审核文案合规性"""
    content = state.get("content", "")
    revision = state.get("revision_count", 0)
    # 简化审核：第1轮通过（真实场景reviewer可能要求修改）
    approved = True
    review_result = f"【审核结果】通过。文案符合广告法，无绝对化用语，功效宣称有成分支撑。审核轮次: {revision + 1}"
    msg = AgentMessage(
        sender="reviewer", receiver="supervisor",
        message_type=MessageType.FEEDBACK,
        content="审核通过，文案可发布",
        metadata={"approved": approved, "revision": revision + 1}
    )
    return {"review_result": review_result, "approved": approved,
            "revision_count": revision + 1, "messages": [msg], "current_agent": "reviewer"}

# 测试4个Agent
test_state = {"task": MARKETING_TASK, "messages": [], "research_data": "", "strategy": "",
              "content": "", "review_result": "", "current_agent": "", "revision_count": 0, "approved": False}
r1 = researcher_agent(test_state); test_state.update(r1)
r2 = strategist_agent(test_state); test_state.update(r2)
r3 = writer_agent(test_state); test_state.update(r3)
r4 = reviewer_agent(test_state); test_state.update(r4)
print("4个营销Agent实现成功")
print(f"  researcher: {r1['research_data'][:40]}...")
print(f"  strategist: {r2['strategy'][:40]}...")
print(f"  writer: {r3['content'][:40]}...")
print(f"  reviewer: approved={r4['approved']}, revision={r4['revision_count']}")
print(f"  消息流: {len(test_state['messages'])}条AgentMessage")

---
## TODO3：用LangGraph构建supervisor中心化拓扑

**supervisor拓扑**（Hub-and-Spoke）：一个supervisor节点负责任务分配和结果汇总，其他Agent各自执行分配的子任务。

```
         supervisor
        /    |    |    \
  researcher strategist writer reviewer
```

用LangGraph实现：
1. `supervisor_node`：根据state的`current_agent`决定下一步路由（researcher->strategist->writer->reviewer->END）
2. 4个Agent节点（TODO2已实现）
3. `add_conditional_edges`：supervisor到各Agent的条件路由
4. Agent执行后返回supervisor（形成星型拓扑）

> 天道推演对应：supervisor=因果链追踪（集中调度），星型拓扑可控但supervisor是单点故障


In [ ]:
# TODO3: 用LangGraph构建supervisor中心化拓扑

def supervisor_node(state: MultiAgentState) -> dict:
    """supervisor节点：记录当前进度，不改变业务字段"""
    return {"current_agent": state.get("current_agent", "init")}

def route_from_supervisor(state: MultiAgentState) -> str:
    """supervisor路由函数：按顺序调度4个Agent"""
    current = state.get("current_agent", "")
    flow = ["init", "researcher", "strategist", "writer", "reviewer"]
    if current in flow:
        idx = flow.index(current)
        if idx + 1 < len(flow):
            return flow[idx + 1]
    return "END"

# 构建supervisor拓扑图
supervisor_builder = StateGraph(MultiAgentState)
supervisor_builder.add_node("supervisor", supervisor_node)
supervisor_builder.add_node("researcher", researcher_agent)
supervisor_builder.add_node("strategist", strategist_agent)
supervisor_builder.add_node("writer", writer_agent)
supervisor_builder.add_node("reviewer", reviewer_agent)

supervisor_builder.add_edge(START, "supervisor")
supervisor_builder.add_conditional_edges(
    "supervisor", route_from_supervisor,
    {"researcher": "researcher", "strategist": "strategist",
     "writer": "writer", "reviewer": "reviewer", "END": END}
)
# 各Agent执行后返回supervisor（星型拓扑）
supervisor_builder.add_edge("researcher", "supervisor")
supervisor_builder.add_edge("strategist", "supervisor")
supervisor_builder.add_edge("writer", "supervisor")
supervisor_builder.add_edge("reviewer", "supervisor")

supervisor_app = supervisor_builder.compile()

print("supervisor中心化拓扑构建成功")
print(f"  节点: supervisor + 4个Agent (researcher/strategist/writer/reviewer)")
print(f"  拓扑: 星型 (hub-spoke), supervisor集中路由")
print(f"  路由: init->researcher->strategist->writer->reviewer->END")
print(f"  设计哲学: 中心化协调，supervisor是流程控制枢纽")

---
## TODO4：用LangGraph构建team去中心化拓扑

**team拓扑**（去中心化）：无中心supervisor，Agent间直接传递消息。每个Agent完成后直接路由到下一个Agent，形成流水线或网状结构。

```
  researcher -> strategist -> writer -> reviewer -> END
```

用LangGraph实现：
1. 无supervisor节点，Agent间直接`add_edge`连接
2. researcher->strategist->writer->reviewer->END（流水线式直接传递）
3. 消息通过State的`messages`字段累积传递（operator.add reducer）

对比supervisor拓扑：去中心化减少了一跳通信，但失去了集中控制能力。

> 天道推演对应：team=自由协作（无中心决策者），需依赖Agent间协议自组织


In [ ]:
# TODO4: 用LangGraph构建team去中心化拓扑

team_builder = StateGraph(MultiAgentState)
team_builder.add_node("researcher", researcher_agent)
team_builder.add_node("strategist", strategist_agent)
team_builder.add_node("writer", writer_agent)
team_builder.add_node("reviewer", reviewer_agent)

# Agent间直接连接（无supervisor中转）
team_builder.add_edge(START, "researcher")
team_builder.add_edge("researcher", "strategist")
team_builder.add_edge("strategist", "writer")
team_builder.add_edge("writer", "reviewer")
team_builder.add_edge("reviewer", END)

team_app = team_builder.compile()

print("team去中心化拓扑构建成功")
print(f"  节点: 4个Agent (无supervisor)")
print(f"  拓扑: 流水线 (pipeline), Agent间直接传递")
print(f"  通信: State.messages累积传递 (operator.add reducer)")
print(f"  设计哲学: 去中心化协作，Agent自组织流程")
print(f"  对比supervisor: 少1个节点，少4条边，但无集中控制")

---
## TODO5：用networkx分析两种Agent通信拓扑

将多Agent系统的通信关系建模为有向图：
- 节点 = Agent（supervisor/researcher/strategist/writer/reviewer）
- 边 = 消息流（谁向谁发消息）

用networkx分析：
1. 构建supervisor拓扑图和team拓扑图
2. `nx.degree_centrality()`：度中心性（谁是通信枢纽）
3. `nx.betweenness_centrality()`：介数中心性（谁是信息瓶颈）
4. `nx.is_strongly_connected()`：强连通性（消息能否到达所有Agent）
5. 识别瓶颈Agent和单点故障风险

> 天道推演对应：networkx指标量化"因果链追踪"--哪个Agent是关键因果节点？


In [ ]:
# TODO5: 用networkx分析两种Agent通信拓扑

# 构建supervisor拓扑图（星型，双向通信）
supervisor_graph = nx.DiGraph()
supervisor_nodes = ["supervisor", "researcher", "strategist", "writer", "reviewer"]
supervisor_graph.add_nodes_from(supervisor_nodes)
# supervisor与每个Agent双向通信
for agent in ["researcher", "strategist", "writer", "reviewer"]:
    supervisor_graph.add_edge("supervisor", agent)  # 任务分配
    supervisor_graph.add_edge(agent, "supervisor")  # 结果汇报

# 构建team拓扑图（流水线，单向传递）
team_graph = nx.DiGraph()
team_nodes = ["researcher", "strategist", "writer", "reviewer"]
team_graph.add_nodes_from(team_nodes)
team_graph.add_edge("researcher", "strategist")
team_graph.add_edge("strategist", "writer")
team_graph.add_edge("writer", "reviewer")

# 计算拓扑指标
print("=" * 70)
print("【networkx拓扑分析】supervisor vs team")
print("=" * 70)

sup_dc = nx.degree_centrality(supervisor_graph)
sup_bc = nx.betweenness_centrality(supervisor_graph)
sup_conn = nx.is_strongly_connected(supervisor_graph)

team_dc = nx.degree_centrality(team_graph)
team_bc = nx.betweenness_centrality(team_graph)
team_conn = nx.is_strongly_connected(team_graph)

print("\n--- supervisor拓扑 ---")
print(f"  节点数: {supervisor_graph.number_of_nodes()}, 边数: {supervisor_graph.number_of_edges()}")
print(f"  度中心性: {dict(sup_dc)}")
print(f"  介数中心性: {dict(sup_bc)}")
print(f"  强连通: {sup_conn}")
print(f"  关键洞察: supervisor度中心性={sup_dc['supervisor']:.2f}(最高), 是通信枢纽和单点故障")

print("\n--- team拓扑 ---")
print(f"  节点数: {team_graph.number_of_nodes()}, 边数: {team_graph.number_of_edges()}")
print(f"  度中心性: {dict(team_dc)}")
print(f"  介数中心性: {dict(team_bc)}")
print(f"  强连通: {team_conn}")
print(f"  关键洞察: strategist介数中心性={team_bc['strategist']:.2f}(最高), 是信息瓶颈")

print("\n--- 拓扑对比 ---")
print(f"  supervisor拓扑: 中心化, supervisor是单点故障, 但集中可控")
print(f"  team拓扑: 去中心化, 无单点故障, 但strategist是瓶颈且不连通(单向)")
print(f"  鲁棒性: team > supervisor (移除supervisor系统崩溃, 移除某Agent仅局部影响)")
print(f"  可控性: supervisor > team (supervisor集中调度, team需自组织)")

---
## TODO6：运行双拓扑系统 + 涌现行为分析 + 天道推演映射

运行supervisor_app和team_app，对比：
1. **执行轨迹**：两种拓扑的Agent调用序列和消息流
2. **涌现指标**：用networkx指标量化涌现质量（通信效率/决策质量/鲁棒性）
3. **天道推演映射**：将多Agent仿真映射到天道推演六能力

> 天道推演核心：多Agent系统是"可计算沙盘"--通过不同拓扑的涌现行为对比，推演哪种拓扑在营销场景下产出更优决策。这把项目CLAUDE.md的「天道推演系统」从思维框架升级为可计算的多Agent仿真工具。


In [ ]:
# TODO6: 运行双拓扑系统 + 涌现行为分析 + 天道推演映射

initial_state = {
    "task": MARKETING_TASK,
    "messages": [],
    "research_data": "",
    "strategy": "",
    "content": "",
    "review_result": "",
    "current_agent": "init",
    "revision_count": 0,
    "approved": False,
}

# === 运行supervisor拓扑 ===
print("=" * 70)
print("【实跑】supervisor中心化拓扑")
print("=" * 70)
sup_result = supervisor_app.invoke(initial_state)
print(f"  调用序列: init -> researcher -> strategist -> writer -> reviewer -> END")
print(f"  消息数: {len(sup_result['messages'])}")
for i, msg in enumerate(sup_result["messages"]):
    print(f"  消息{i+1}: {msg.sender}->{msg.receiver} [{msg.message_type.value}] {msg.content[:35]}...")
print(f"  调研: {sup_result['research_data'][:50]}...")
print(f"  策略: {sup_result['strategy'][:50]}...")
print(f"  文案: {sup_result['content'][:50]}...")
print(f"  审核: {sup_result['review_result'][:50]}...")
print(f"  通过: {sup_result['approved']}")

# === 运行team拓扑 ===
print("\n" + "=" * 70)
print("【实跑】team去中心化拓扑")
print("=" * 70)
team_result = team_app.invoke(initial_state)
print(f"  调用序列: researcher -> strategist -> writer -> reviewer -> END")
print(f"  消息数: {len(team_result['messages'])}")
for i, msg in enumerate(team_result["messages"]):
    print(f"  消息{i+1}: {msg.sender}->{msg.receiver} [{msg.message_type.value}] {msg.content[:35]}...")
print(f"  调研: {team_result['research_data'][:50]}...")
print(f"  策略: {team_result['strategy'][:50]}...")
print(f"  文案: {team_result['content'][:50]}...")
print(f"  审核: {team_result['review_result'][:50]}...")
print(f"  通过: {team_result['approved']}")

# === 涌现行为分析 ===
print("\n" + "=" * 70)
print("【涌现行为分析】多Agent仿真 × 天道推演")
print("=" * 70)
print(f"  supervisor拓扑: {len(sup_result['messages'])}条消息, 5个节点(supervisor+4Agent), 8条边")
print(f"  team拓扑:       {len(team_result['messages'])}条消息, 4个节点(4Agent), 3条边")
print(f"  通信效率: team > supervisor (少1跳中转, 消息直达)")
print(f"  决策质量: 两者产出相同策略 (离线StubLLM), 真实LLM下supervisor可动态重路由")
print(f"  鲁棒性:   team > supervisor (supervisor是单点故障)")
print(f"  可控性:   supervisor > team (集中调度, 可条件重路由)")

# === 天道推演六能力映射 ===
print("\n--- 天道推演 × 多Agent仿真 六能力映射 ---")
print(f"  局势感知    -> researcher_agent: 感知市场棋盘(产品/竞品/用户)")
print(f"  因果链追踪  -> AgentMessage: 记录sender->receiver因果链, networkx介数中心性识别关键节点")
print(f"  沙盘模拟    -> StateGraph: 构建多Agent沙盘, 条件边展开多分支执行")
print(f"  概率评估    -> networkx中心性: 量化各Agent的通信枢纽度和瓶颈风险")
print(f"  最优路径推荐 -> supervisor路由: 选择最优Agent协作序列(researcher->strategist->writer->reviewer)")
print(f"  反馈学习    -> reviewer_agent: 审核反馈+revision_count, 支持多轮修改循环")

print("\n--- 多Agent仿真结论 ---")
print(f"  涌现质量: 多Agent系统整体行为优于单Agent (专业化分工 + 通信协议 + 审核闭环)")
print(f"  拓扑选型: 营销流程明确 -> supervisor拓扑 (可控); 探索性任务 -> team拓扑 (灵活)")
print(f"  天道推演升级: 从'个人思维框架'升级为'可计算多Agent沙盘', 用networkx量化推演拓扑质量")
print(f"  2026前沿: A2A协议标准化Agent间通信, MCP标准化Agent工具连接, 多Agent仿真成为决策工具")